# 10 — Sequential DoE: augment and propose next runs

Close the loop: start from a weak design, propose the next batch, and ask whether the extra runs are worth it (Δ D/G/SPV/power).

Desde **0.7**, el mismo flujo se expone como `ed.experiment(...).ingest(y).next(...)`.


In [1]:
import numpy as np
import doekit as ed
print("doekit", ed.__version__)

doekit 0.7.0


## 1. Weak starting design

In [2]:
facs = [ed.ContinuousFactor("x1", -1, 1), ed.ContinuousFactor("x2", -1, 1)]
base = ed.random_design(facs, n=6, seed=0)
base.model = ed.Model.parse("0 ~ x1 + x2 + x1:x2")
print(ed.evaluate(base, n_region=2000, seed=0).summary())

Design evaluation  (6 runs, 4 params, dof=2)
--------------------------------------------------------
  D-efficiency :   36.3 %
  A-efficiency :   21.4 %
  G-efficiency :    9.0 %
  SPV (scaled prediction variance) over region:
      min=1.191  mean=5.964  max=44.239
  Power (effect/sigma anticipated):
      (Intercept)                   0.19
      x1                            0.12
      x2                            0.10
      x1:x2                         0.09
  Max VIF: 2.03


## 2. Propose next runs (no response yet)

In [3]:
prop = ed.propose_next_runs(base, n_add=4, criterion="D", n_candidates=120, seed=1)
print(prop.rationale)
print(prop.comparison.summary)
display(prop.comparison.table)
display(prop.added.matrix)

Propose 4 new run(s) by D-optimal augmentation of the current 6-run design. Yes: 4 extra run(s) (ΔD=+37.4 pts, ΔG=+40.7 pts, ΔSPV_mean=-3.57, Δpower=+0.480).
Yes: 4 extra run(s) (ΔD=+37.4 pts, ΔG=+40.7 pts, ΔSPV_mean=-3.57, Δpower=+0.480).


,metric,current,augmented,delta
0,n_runs,6.000000,10.000000,4.000000
1,D_efficiency,36.347791,73.770188,37.422397
2,A_efficiency,21.372193,69.512324,48.140131
3,G_efficiency,9.157285,49.854790,40.697505
4,spv_mean,5.861761,2.289362,-3.572400
5,mean_power,0.123528,0.603706,0.480178


,x1,x2
0,-0.966945,0.870145
1,-0.966945,-0.994523
2,0.825511,0.870145
3,0.825511,-0.994523


## 3. With simulated responses

In [4]:
rng = np.random.default_rng(2)
X = base.model.matrix(base.matrix)
beta = np.array([1.0, 2.0, -1.5, 0.8])
y = X @ beta + rng.normal(0, 0.3, base.n_runs)
prop2 = ed.propose_next_runs(base, response=y, n_add=4, budget=16, seed=3)
print("sigma_hat", prop2.sigma_hat)
print("active", prop2.active_terms)
print(prop2.comparison.summary)
print(prop2.to_dict()["schema"])

sigma_hat 0.16581886645557764
active ['x1', 'x2', 'x1:x2']
Yes: 4 extra run(s) (ΔD=+37.4 pts, ΔG=+40.5 pts, ΔSPV_mean=-3.62, Δpower=+0.108).
doekit.NextRunsProposal/1


## 4. BO bridge (bounds → candidates)

In [5]:
cand = ed.candidates_from_bounds([("x1", -1, 1), ("x2", -1, 1)], n=100, seed=4)
aug = ed.augment_design(base, n_add=3, candidates=cand, criterion="I", seed=5)
print(ed.compare_designs(base, aug, n_region=1000, seed=0).summary)

Yes: 3 extra run(s) (ΔD=+22.0 pts, ΔG=+36.3 pts, ΔSPV_mean=-3.29, Δpower=+0.336).


## Puente: mismo flujo con `Experiment`

Façade de producto sobre las primitivas de arriba: evaluate → ingest → next → snapshot.


In [6]:
exp = ed.experiment(design=base, model=base.model)
exp.evaluate(n_region=800, seed=0)
exp.ingest(y)
nxt = exp.next(n_add=4, budget=16, n_candidates=80, n_starts=2, seed=3)
print(nxt.comparison.summary)
print("schema:", exp.to_dict()["schema"])
print("CSV:", exp.export_csv("reports/10_sequential/runs.csv"))


Yes: 4 extra run(s) (ΔD=+37.4 pts, ΔG=+40.5 pts, ΔSPV_mean=-3.62, Δpower=+0.108).
schema: doekit.Experiment/1
CSV: reports\10_sequential\runs.csv
